In [11]:
import arcpy
import os
import time
import re

In [12]:
# Uncheck the World Topographic Map and World Hillshade in content pane before running the script.
# Update the path and Utility base upon data.

utility = "UNION"
map_type = "LANDBASE"
package_dir = r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_"
wkid_to_set = 3857 # The WKID for WGS_1984_Web_Mercator_Auxiliary_Sphere is 3857

# Update scale for all the layers
# Landbase scale 1:1000000 - 1
# Network scale 1:1000000 - 50000

default_min_scale = 10000
default_max_scale = 1

label_min_scale = 100000
label_max_scale = 1 

layers_to_skip = [
    "World Topographic Map",
    "World Hillshade",
    "Basemap"
]
print(f"Parameter set for {utility} {map_type} with {wkid_to_set} coordinate")


Parameter set for UNION LANDBASE with 3857 coordinate


In [13]:
# Set the custom scales for the layers

def set_layer_scale(layer_list):
    target_text = "STREETS"
    if layer_list is None:
        return

    for layer in layer_list:
        try:
            # Check if the layer's name is skip list
            if layer.name in layers_to_skip:
                print(f"\nSkipping Base Maps: {layer.name}")
                continue  # This moves to the next layer in the loop
            
            if layer.supports("minThreshold") and layer.supports("maxThreshold"):
                # print(f"Setting scale for: {layer.name}")
                # layer.minThreshold = default_min_scale
                # layer.maxThreshold = default_max_scale

                if target_text.lower() in layer.name.lower():
                    print(f"Setting scale for: {layer.name} with {label_min_scale}")
                    layer.minThreshold = label_min_scale
                    layer.maxThreshold = label_max_scale
                # else:
                #     print(f"Setting scale for: {layer.name} with {default_min_scale}")
                #     layer.minThreshold = default_min_scale
                #     layer.maxThreshold = default_max_scale
            else:
                print(f"\nSkipping (not supported): {layer.name}")

            # If the layer is a group layer, recursively call this function
            # if layer.isGroupLayer:
            #     print(f"\n--- Entering Group: {layer.name} ---\n")
            #     set_layer_scale(layer.listLayers())
                # print(f"--- Exiting Group: {layer.name} ---")

        except Exception as e:
            print(f"Error setting scale for {layer.name}: {e}")
            
print("Completed running 'Set_Layer_scale function'")

Completed running 'Set_Layer_scale function'


In [14]:
# Finds all unique codes in the layer names within a given map
   
def extract_layer_codes(name_list):
    """
    Args:
        map_names(Name of the map consisting of codes e.g New_data_ON01)

    Returns:
        set: A set of all unique codes found.
    """
    results = []
    
    # Loop through each item in the input list
    for text in name_list:
        # Use re.split to break the string apart by the underscore '_'
        parts = re.split(r"_", text) 
        
        # Get the last element of the resulting list
        last_segment = parts[-1] 
        
        # Add the result to the output list
        results.append(last_segment)
        
    return results
       
print("Completed running function 'Extract Layer Codes'")

Completed running function 'Extract Layer Codes'


In [15]:
# Get the current ArcGIS Pro project and the active map.
try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
        
    print(f"The active map is: {active_map.name}\n")

    # Set the path to your project's geodatabase for the output.
    # workspace_gdb = r"C:\path\to\your\project.gdb"
    # maps = aprx.listMaps()

    if not active_map:
        print(f"No Maps found in the project.")
    else:
        # Creating a SpatialReference object from the WKID
        new_cs = arcpy.SpatialReference(wkid_to_set)

        # Set the map's spatial reference
        active_map.spatialReference = new_cs

        print(f"Successfully set map CS to: {active_map.spatialReference.name}\n")
        
        # Get the list of layers from the map
        # set_layer_scale(active_map.listLayers())
        print(f"\n ----- Scale Updated.-----")        

     # Get the layers from the map
    all_layers_list = active_map.listLayers()
    
    if not all_layers_list:
        print("Error: No layers found in the active map.")
    else:
        # Flag to track if we find a valid layer
        found_valid_layer = False 
        
        first_layer = [all_layers_list[0].name]
        print(f"The first Layer in TOC is : {first_layer}")

        map_names = extract_layer_codes(first_layer)
        
        old_map_name = active_map.name
        new_map_name = list(map_names)[0]

        print(f"\nRenaming active map from '{old_map_name}' to '{new_map_name}'...")
        active_map.name = new_map_name

        map_view = aprx.activeView

        thumb_loc = package_dir + f"{utility}_{map_type}_{active_map.name}"+".png"
        print(f"\nExporting thumbnail to {thumb_loc}...")
        
        map_view.exportToPNG(thumb_loc, width=600, height=400)
        
        # Loop through layers to find the first one with a data source
        for layer in all_layers_list:
            
            # Check if the layer is a data layer (e.g., not a Group Layer)
            if layer.supports("DATASOURCE"):
                
                # We found a valid layer!
                source_layer = layer
                found_valid_layer = True                
                # print(f"Found first valid data layer: {source_layer.name}")
                
                # Get the full extent from the layer's data source ---
                desc = arcpy.Describe(source_layer.dataSource)
                source_extent = desc.extent
                
                # Use the extent by setting the environment ---
                arcpy.env.extent = source_extent
                
                print(f"Successfully set extent to layer: {source_layer.name}")
                print(f"New extent: {arcpy.env.extent}")
                # print(f"Source extent object: {source_extent}")
                
                # Exit the loop 
                break
            
            else:
                # This layer was skipped (e.g., it's a Group Layer)
                print(f"\nSkipping layer (not a data layer): {layer.name}")

        # If the loop finished without finding any valid layers
        if not found_valid_layer:
            print("Error: No layers with a valid data source were found in the map.")
    
    workspace = aprx.defaultGeodatabase
    # print(f"Project's Default Geodatabase: {workspace}")

    # Set the default geodatabase
    if workspace and arcpy.Exists(workspace):
        arcpy.env.workspace = workspace
        print(f"\nWorkspace has been set to: {arcpy.env.workspace}")
        
    else:
        print("\nCould not find a valid geodatabase to set the workspace.")

except Exception as e:
    print(f"Error accessing the project or active map: {e}")
    # exit if no map is active.
    exit()


The active map is: Union Landbase HN01

Successfully set map CS to: WGS_1984_Web_Mercator_Auxiliary_Sphere


 ----- Scale Updated.-----
The first Layer in TOC is : ['Union Landbase HN01']

Renaming active map from 'Union Landbase HN01' to 'Union Landbase HN01'...

Exporting thumbnail to C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_UNION_LANDBASE_Union Landbase HN01.png...

Skipping layer (not a data layer): Union Landbase HN01
Error accessing the project or active map: "Server=pvsrv-postgres01,Instance=esri,Database Platform=PostgreSQL,User=gis,Version=sde.DEFAULT (Traditional),Authentication Type=Database Authentication,Dataset=esri.gis.union_landbase_address" does not exist


<class 'AttributeError'>: 'XPythonShell' object has no attribute 'ask_exit'

In [19]:
# Name for the output index feature class that will be created.
output_index_features = f"{utility}_{map_type}_{active_map.name}_Tile_Index"
output_vtpk_package = package_dir + f"{utility}_{map_type}_{active_map.name}"+".vtpk" 

try:   
    # Construct the full path for the output.
    out_features_path = os.path.join(workspace, output_index_features)
    # print(out_features_path)
    
    # --- Record the start time ---
    script_start_time = time.perf_counter()
    print(f"\nStarting to create vector tile index for the map: '{active_map.name}'...")

   # Execute the Create Vector Tile Index tool using the active map as input.
    arcpy.management.CreateVectorTileIndex(
        in_map=active_map,
        # out_featureclass=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\OW01_VTPK_1M_500_test.shp",
        out_featureclass=out_features_path,
        service_type="ONLINE",
        tiling_scheme=None,
        vertex_count=10000,
    )

    print(f"Successfully created index features at: {out_features_path}\n")

    # Remove the tile index layer from the layerlist before creating the vector tile package.
    remove_layer_name = output_index_features

    layer_to_remove = None
    
    for lyr in active_map.listLayers():
        if lyr.name == remove_layer_name:
            layer_to_remove = lyr
            break
    
    if layer_to_remove:
        active_map.removeLayer(layer_to_remove)
        print(f"Successfully removed layer from content: '{remove_layer_name}'\n")
    else:
        print(f"Error: Layer '{remove_layer_name}' not found in the map '{active_map.name}'.")

    # Creating a vector tile package using the index feature 
    print(f"Starting to create vector tile package for the map: '{active_map.name}'...")
    
    arcpy.management.CreateVectorTilePackage(
        in_map=active_map,
        output_file= output_vtpk_package,
        service_type="ONLINE",
        tiling_scheme=None,
        tile_structure="INDEXED",
        min_cached_scale=144448,
        max_cached_scale=282,
        # index_polygons=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\Alektra_HN01_Network_VTPK_Index_Feature.shp",
        index_polygons= out_features_path,
        summary="",
        tags=""
    )

    print(f"Successfully created Vector Tile Package at: {output_vtpk_package}")

except arcpy.ExecuteError:
    # Get and print geoprocessing error messages.
    print(arcpy.GetMessages(2))

except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")

finally:
    # --- 3. This block always runs (success or error) ---
    script_end_time = time.perf_counter()
    elapsed_time = script_end_time - script_start_time
    
    # Optional: Convert to minutes and seconds for readability
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print("\n---------------------------------")
    print(f"Script finished in {minutes} minutes and {seconds:.2f} seconds.")
    print(f"(Total elapsed: {elapsed_time:.2f} seconds)")



Starting to create vector tile index for the map: 'HN01'...
ERROR 003981: The server took too long to answer. The client has timed out.
Failed to execute (CreateVectorTileIndex).


---------------------------------
Script finished in 6 minutes and 38.78 seconds.
(Total elapsed: 398.78 seconds)


In [12]:
# This creates a Hosted Vector Tile Layer
from arcgis.gis import GIS

# Connect to your ArcGIS Online or Portal
# In an ArcGIS Notebook, "home" connects automatically
gis = GIS("home")
user = gis.users.me

# Update the name of the folder from ArcGIS Enterprise/online
folder_name = "VTPK"

# Set the item properties for the new item in your portal
item_properties = {
    "title": f"{utility}_{map_type}_{active_map.name}_VTPK",
    "tags": f"Vector, Tiles, {active_map.name}, {map_type}, {utility}",
    "type": "Vector Tile Package",
    "description": f"{utility} {map_type} {active_map.name} Hosted Vector Tile Layers."
}
# print(item_properties)

# Add the .vtpk to your portal as an item
print("Uploading the .vtpk package...")

vtpk_item = gis.content.add(item_properties, 
                            data=output_vtpk_package,
                            folder=folder_name)

# # Publish the item to create the hosted layer
print("Publishing hosted vector tile layer...")
hosted_tile_layer_item = vtpk_item.publish()

print(f"\nSuccessfully published layer: {hosted_tile_layer_item.url}")

Uploading the .vtpk package...


C:\Users\sameer.bajracharya\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\IPython\core\interactiveshell.py:3550: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)
C:\Users\sameer.bajracharya\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gis-sandbox.planview.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Publishing hosted vector tile layer...

Successfully published layer: https://gis-sandbox.planview.ca/server/rest/services/Hosted/BELLONTARIO_LANDBASE_T01_VTPK/VectorTileServer


In [13]:
# Set Extent and Update Thumbnail
print("\nSetting Extent and Thumbnail...")

api_extent = [
    source_extent.XMin,
    source_extent.YMin,
    source_extent.XMax,
    source_extent.YMax,
]

# custom extent
# api_extent = [-9331983.83, 5001946.96, -8489798.92, 5763191.07]

print(api_extent)

print(f"Setting item's bounding box (extent)...")
hosted_tile_layer_item.update(
    item_properties={
        "extent": api_extent
    },
    thumbnail=thumb_loc
)
        
print("--- Thumbnail updated successfully! ---")
print(f"Published layer Link: {hosted_tile_layer_item.url}")


Setting Extent and Thumbnail...
[-95.15295537099996, 41.739624941000045, -74.38110375899998, 53.84318174800006]
Setting item's bounding box (extent)...
--- Thumbnail updated successfully! ---
Published layer Link: https://gis-sandbox.planview.ca/server/rest/services/Hosted/BELLONTARIO_LANDBASE_T01_VTPK/VectorTileServer


In [77]:
# Testing code below this

import arcpy

try:
    # Get the project and active map
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
    
    # Get a list of ALL layers in the map
    all_layers_list = active_map.listLayers()    
    
    if not all_layers_list:
        print("Error: No layers found in the active map.")
    else:
        # Flag to track if we find a valid layer
        found_valid_layer = False 
        first_layer = [all_layers_list[0].name]
        print(f"The first Layer in TOC is : {first_layer}")

        map_names = extract_layer_codes(first_layer)
        print(test)

        old_map_name = active_map.name
        new_map_name = list(map_names)[0]

        print(f"Renaming active map from '{old_map_name}' to '{new_map_name}'...")
        active_map.name = new_map_name
        
        # Loop through layers to find the first one with a data source
        for layer in all_layers_list:
            
            # Check if the layer is a data layer (e.g., not a Group Layer)
            if layer.supports("DATASOURCE"):
                
                # We found a valid layer!
                source_layer = layer
                found_valid_layer = True
                
                print(f"Found first valid data layer: {source_layer.name}")
                
                # Get the full extent from the layer's data source ---
                desc = arcpy.Describe(source_layer.dataSource)
                source_extent = desc.extent
                
                # Use the extent by setting the environment ---
                arcpy.env.extent = source_extent
                
                print(f"Successfully set geoprocessing extent to layer: {source_layer.name}")
                print(f"New extent: {arcpy.env.extent}")
                print(f"Source extent object: {source_extent}")
                
                # Exit the loop 
                break
            
            else:
                # This layer was skipped (e.g., it's a Group Layer)
                print(f"Skipping layer (not a data layer): {layer.name}")

        # If the loop finished without finding any valid layers
        if not found_valid_layer:
            print("Error: No layers with a valid data source were found in the map.")

except Exception as e:
    print(f"An error occurred: {e}")



The first Layer in TOC is : ['HYDRO_ONE_NETWORK_T01']
{'T01'}
Renaming active map from 'ONE01' to 'T01'...
Skipping layer (not a data layer): HYDRO_ONE_NETWORK_T01
Found first valid data layer: TRANSFORMER
Successfully set geoprocessing extent to layer: TRANSFORMER
New extent: -95.15057347 41.7360082500001 -74.3444280959999 56.0188775220001 NaN NaN NaN NaN
Source extent object: -95.15057347 41.7360082500001 -74.3444280959999 56.0188775220001 NaN NaN NaN NaN


In [15]:
# --- This is the new code to add ---
print("Layer published. Now updating item extent...")
print(arcpy.env.extent)

try:
    # 1. Get the actual layer service object from the item
    layer_service = hosted_tile_layer_item.layers[0]
    
    # 2. Get the service's full extent (it's a dictionary)
    service_extent_dict = layer_service.properties.fullExtent
    
    if service_extent_dict:
        # 3. Format it as a list: [xmin, ymin, xmax, ymax]
        extent_list = [
            service_extent_dict['xmin'],
            service_extent_dict['ymin'],
            service_extent_dict['xmax'],
            service_extent_dict['ymax']
        ]

        print(extent_list)
        
        # 4. Update the portal item's metadata
        # hosted_tile_layer_item.update(
        #     item_properties={
        #         "extent": extent_list
        #     }
        # )
        print("Successfully updated item extent.")
    else:
        print("Warning: Could not read extent from layer service.")

except Exception as e:
    print(f"Error updating extent: {e}")

# Now when you open this URL, it should zoom correctly
print(f"Layer URL: {hosted_tile_layer_item.url}")

Layer published. Now updating item extent...
-95.151115843 42.192283424 -74.451621113 49.853532919 NaN NaN NaN NaN
[-10592173.76405454, 5189826.228308224, -8287916.551032894, 6420948.85516627]
Successfully updated item extent.
Layer URL: https://gis-sandbox.planview.ca/server/rest/services/Hosted/Telus_Network_Test_Hosted_Vector_Tile_Layer/VectorTileServer


In [17]:
import arcpy
from arcgis.gis import GIS

# Define the target spatial reference (Web Mercator) ---
# This is what ArcGIS Online/Portal map viewers use by default
web_mercator_sr = arcpy.SpatialReference(3857)

# Get the extent object from the arcpy environment
my_arcpy_extent = arcpy.env.extent
print(my_arcpy_extent)
if my_arcpy_extent:
    try:
        # --- 3. CRITICAL STEP: Project the extent to Web Mercator ---
        print("Projecting local extent to Web Mercator (WKID 3857)...")
        projected_extent = my_arcpy_extent.projectAs(web_mercator_sr)
        
        # --- 4. Convert the *projected* extent to a list ---
        extent_list = [
            projected_extent.XMin,
            projected_extent.YMin,
            projected_extent.XMax,
            projected_extent.YMax
        ]

        # --- 5. Update the hosted item's metadata ---
        # (Make sure 'hosted_tile_layer_item' is your published service item)
        print(f"Updating item extent with projected coordinates: {extent_list}")
        hosted_tile_layer_item.update(
            item_properties={
                "extent": extent_list
            }
        )
        
        print("Successfully updated item extent.")
        print(f"View your layer here: {hosted_tile_layer_item.url}")
        
    except Exception as e:
        print(f"Error projecting extent. Is your original extent's spatial reference defined? Error: {e}")

else:
    print("Error: arcpy.env.extent is not set. Cannot update item.")

-95.151115843 42.192283424 -74.451621113 49.853532919 NaN NaN NaN NaN
Projecting local extent to Web Mercator (WKID 3857)...
Updating item extent with projected coordinates: [-10592173.76405454, 5189826.228308224, -8287916.551032894, 6420948.85516627]
Successfully updated item extent.
View your layer here: https://gis-sandbox.planview.ca/server/rest/services/Hosted/Telus_Network_Test_Hosted_Vector_Tile_Layer/VectorTileServer


In [47]:
try:
    # Get a list of all feature classes in the workspace
    feature_classes = arcpy.ListFeatureClasses()
    if not feature_classes:
        print(f"No feature classes found in: {arcpy.env.workspace}")
    else:
        #  Record the start time ---
        script_start_time = time.perf_counter()
        
        print(f"--- Projections in: {arcpy.env.workspace} ---")
        
        # Loop through each feature class in the list
        for fc in feature_classes:
            try:
                # Get the describe object
                desc = arcpy.Describe(fc)
                
                # Get the spatial reference object
                spatial_ref = desc.spatialReference
                
                # Print the feature class name
                print(f"\nLayer: {fc}")
                
                # Check if a coordinate system is defined
                if spatial_ref.name == "Unknown":
                    print("  Projection: ** UNKNOWN **")
                else:
                    # Print the name and WKID (Factory Code)
                    print(f"  Name: {spatial_ref.name}")
                    print(f"  WKID: {spatial_ref.factoryCode}")
                    print(f"  Type: {spatial_ref.type}")

            except Exception as e:
                print(f"\nError describing {fc}: {e}")

except arcpy.ExecuteError:
    print(f"ArcPy Error: Could not access workspace '{arcpy.env.workspace}'.")
    print(arcpy.GetMessages(2))
except Exception as e:
    print(f"A general error occurred: {e}")

finally:
    # --- 3. This block always runs (success or error) ---
    script_end_time = time.perf_counter()
    elapsed_time = script_end_time - script_start_time
    
    # Optional: Convert to minutes and seconds for readability
    minutes = int(elapsed_time // 60)
    seconds = elapsed_time % 60
    
    print("---------------------------------")
    print(f"Script finished in {minutes} minutes and {seconds:.2f} seconds.")
    print(f"(Total elapsed: {elapsed_time:.2f} seconds)")

--- Projections in: C:\Users\sameer.bajracharya\Documents\ArcGIS\Packages\Alectra_w_definition_queries_1a3bd1\commondata\Publishing.gdb ---

Layer: OW01_VTPK_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: HN01_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: GE01_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: Telus_GE01_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: Telus_GN01_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: Telus_GN02_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: Telus_GW01_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliary_Sphere
  WKID: 3857
  Type: Projected

Layer: Telus_GW02_Landbase_Tile_Index
  Name: WGS_1984_Web_Mercator_Auxiliar

In [20]:
import arcpy

try:
    # Get the current ArcGIS Pro project and active map
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
    
    if active_map:
        print(f"--- Layer Projections in Map: {active_map.name} ---")
        
        # Get a list of all layers in the map
        layers = active_map.listLayers()
        
        if not layers:
            print("This map contains no layers.")
        else:
            for lyr in layers:
                # We only check feature layers (not group layers, etc.)
                if lyr.isFeatureLayer:
                    try:
                        # Describe the layer's data source to get its properties
                        desc = arcpy.Describe(lyr.dataSource)
                        spatial_ref = desc.spatialReference
                        
                        print(f"\nLayer: {lyr.name}")
                        
                        if spatial_ref.name == "Unknown":
                            print("  Projection: ** UNKNOWN **")
                        else:
                            print(f"  Name: {spatial_ref.name}")
                            print(f"  WKID: {spatial_ref.factoryCode}")
                    
                    except Exception as e:
                        print(f"\nCould not get projection for layer: {lyr.name}")
                        print(f"  Error: {e}")
                else:
                    print(f"\nLayer: {lyr.name} (Not a feature layer)")

    else:
        print("No active map found. Please open a map view.")

except Exception as e:
    print(f"An error occurred: {e}")

--- Layer Projections in Map: Test ---

Could not get projection for layer: telus_landbase_parcels
  Error: "Server=pvsrv-postgres01,Instance=esri,Database Platform=PostgreSQL,User=gis,Version=sde.DEFAULT (Traditional),Authentication Type=Database Authentication,Dataset=esri.gis.telus_landbase_parcels" does not exist

Layer: World Topographic Map (Not a feature layer)

Layer: World Hillshade (Not a feature layer)


In [22]:
import arcpy
import os

# --- 1. Set up the Environment ---
try:
    # Get the current ArcGIS Pro project
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    
    # Get the active map
    m = aprx.activeMap
    
    # Get the project's default geodatabase
    out_gdb = aprx.defaultGeodatabase
    
    # Define the output coordinate system (WKID 3857)
    out_cs = arcpy.SpatialReference(3857) # WGS_1984_Web_Mercator_Auxiliary_Sphere

    print(f"Projecting layers to: {out_cs.name}")
    print(f"Output location: {out_gdb}\n")

    # --- 2. Loop Through Layers and Project ---
    if not m:
        print("No active map found. Please open a map and try again.")
    else:
        for lyr in m.listLayers():
            if not lyr.isFeatureLayer:
                print(f"Skipping '{lyr.name}' (not a feature layer).")
                continue

            print(f"Processing layer: {lyr.name}")
            
            try:
                # Get the layer's source coordinate system
                desc = arcpy.Describe(lyr.dataSource)
                in_cs = desc.spatialReference
                
                # Skip if already in the correct projection
                if in_cs.factoryCode == out_cs.factoryCode:
                    print(f"  > Already in {out_cs.name}. Skipping.")
                    continue

                # Define the output feature class name
                out_name = f"{lyr.name}_WebMercator"
                out_fc = os.path.join(out_gdb, out_name)

                # --- 3. Find Transformation ---
                # This is the most critical step
                transformations = arcpy.ListTransformations(in_cs, out_cs)
                
                transformation_name = ""
                if transformations:
                    # Use the first (often best) transformation found
                    transformation_name = transformations[0]
                    print(f"  > Using transformation: {transformation_name}")
                else:
                    # This can happen if no transformation is needed or available
                    print(f"  > No specific transformation found or needed.")

                # --- 4. Run the Project Tool ---
                arcpy.management.Project(
                    in_dataset=lyr.dataSource,
                    out_dataset=out_fc,
                    out_coor_system=out_cs,
                    transform_method=transformation_name
                )
                
                print(f"  > SUCCESS: Created '{out_name}'")

            except arcpy.ExecuteError:
                print(f"  > ERROR: Could not project '{lyr.name}'.")
                print(arcpy.GetMessages(2))
            except Exception as e:
                print(f"  > ERROR: An unexpected error occurred with '{lyr.name}': {e}")
        
        print("\n--- Projection complete. ---")

except Exception as e:
    print(f"A general error occurred: {e}")

Projecting layers to: WGS_1984_Web_Mercator_Auxiliary_Sphere
Output location: C:\Users\sameer.bajracharya\Documents\ArcGIS\Packages\Alectra_w_definition_queries_1a3bd1\commondata\Publishing.gdb

Skipping 'TELUS_NETWORK_GN01' (not a feature layer).
Processing layer: VAULT
  > No specific transformation found or needed.
  > ERROR: Could not project 'VAULT'.
Failed to execute. Parameters are not valid.
ERROR 000185: This data type is not supported
Failed to execute (Project).

Processing layer: SERVICE_BOX
  > No specific transformation found or needed.
  > ERROR: Could not project 'SERVICE_BOX'.
Failed to execute. Parameters are not valid.
ERROR 000185: This data type is not supported
Failed to execute (Project).

Processing layer: POLE
  > No specific transformation found or needed.
  > ERROR: Could not project 'POLE'.
Failed to execute. Parameters are not valid.
ERROR 000185: This data type is not supported
Failed to execute (Project).

Processing layer: MANHOLE
  > No specific transfo

In [ ]:
# Finds all unique codes in the layer names within a given map.

import re

def extract_layer_codes(name_list, pattern_regex=r"[A-Z]+\d+"):
    """
    Args:
        map_names(Name of the map consisting of codes e.g New_data_ON01)

    Returns:
        set: A set of all unique codes found.
    """
    pattern = re.compile(pattern_regex)
    unique_codes = set()
    
    for name in name_list:
        match = pattern.search(name)
        if match:
            unique_codes.add(match.group(0))
            
    return unique_codes

In [92]:
# Change the coordinate system
import arcpy

# The WKID for WGS_1984_Web_Mercator_Auxiliary_Sphere is 3857
wkid_to_set = 3857

try:
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    active_map = aprx.activeMap
    
    if not active_map:
        print("Error: No active map found.")
    else:
        # 1. Create a SpatialReference object from the WKID
        new_cs = arcpy.SpatialReference(wkid_to_set)
        
        # 2. Set the map's spatial reference
        active_map.spatialReference = new_cs
        
        print(f"Successfully set map CS to: {active_map.spatialReference.name}")
        print(f"WKID: {active_map.spatialReference.factoryCode}")

except Exception as e:
    print(f"An error occurred: {e}")

Successfully set map CS to: WGS_1984_Web_Mercator_Auxiliary_Sphere
WKID: 3857


In [29]:
import arcpy
import os
import time

# --- 1. Get the current project ---
aprx = arcpy.mp.ArcGISProject("CURRENT")

# --- 2. Set your output path ---
project_folder = aprx.homeFolder 
output_thumbnail = os.path.join(project_folder, "map_thumbnail.png")

arcpy.AddMessage(f"Saving to: {output_thumbnail}")

# --- 3. Get the active view ---
view = aprx.activeView

# --- 4. Export the view ---
if view and hasattr(view, 'map'):
    arcpy.AddMessage(f"Exporting view of '{view.map.name}'...")

    # --- FIX: Force the view to redraw ---
    arcpy.AddMessage("Forcing map redraw...")
    cam = view.camera
    original_scale = cam.scale

    # 1. "Nudge" the camera by 1% to force a redraw
    cam.scale = original_scale * 1.01
    view.camera = cam

    # 2. Give Pro time to *complete* the redraw (increased to 2 seconds)
    time.sleep(2) 

    # 3. Now, export the view
    view.exportToPNG(output_thumbnail, width=600, height=400)
    
    # 4. (Optional but good) Set the camera back
    cam.scale = original_scale
    view.camera = cam
    
    arcpy.AddMessage("Export successful.")
    
elif view and hasattr(view, 'layout'):
    arcpy.AddError("Error: Active view is a Layout.")
    
else:
    arcpy.AddError("Error: No active view found.")